# 👤 Human-in-the-Loop

## What is Human-in-the-Loop?
Pause the graph execution and **wait for human input** before continuing.

## Use Cases
- Approve before executing dangerous actions
- Request clarification when LLM is uncertain
- Review generated code before running
- Manual data entry at specific points

## Implementation: interrupt_before
```python
app = graph.compile(
    checkpointer=checkpointer,
    interrupt_before=['dangerous_tool']  # pause before this node
)
```


In [ ]:
# ── Human Approval Pattern ─────────────────────────────────────────────
from typing import TypedDict, Annotated
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

class ReviewState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]
    action: str
    approved: bool

def generate_action(state: ReviewState) -> dict:
    action = 'DELETE all users from database'
    return {
        'action': action,
        'messages': [AIMessage(content=f'Proposed action: {action}')]
    }

def execute_action(state: ReviewState) -> dict:
    if state['approved']:
        return {'messages': [AIMessage(content=f'✅ Executed: {state["action"]}')]}
    else:
        return {'messages': [AIMessage(content='❌ Action cancelled by user')]}

checkpointer = MemorySaver()
graph = StateGraph(ReviewState)
graph.add_node('generate', generate_action)
graph.add_node('execute', execute_action)
graph.set_entry_point('generate')
graph.add_edge('generate', 'execute')
graph.add_edge('execute', END)

# interrupt_before: pause BEFORE execute node
app = graph.compile(checkpointer=checkpointer, interrupt_before=['execute'])

config = {'configurable': {'thread_id': 'approval_001'}}

# Run until interruption
result = app.invoke({'messages': [], 'action': '', 'approved': False}, config)
print('Graph paused. Current state:')
print(f'  Proposed action: {result["action"]}')
print('  Waiting for human approval...')

# Simulate human approval
print('\n[HUMAN]: Approving action...')
app.update_state(config, {'approved': True})

# Resume
final = app.invoke(None, config)
print('\nFinal message:', final['messages'][-1].content)

## 🎯 Interview Questions

1. **What is human-in-the-loop in LangGraph?**
2. **How do you pause a graph for human review?**
3. **What is interrupt_before and interrupt_after?**
4. **How do you resume a paused graph?**

> Answer these before moving to the next module.

## 💪 Mini Exercise

**Exercise**: Based on what you learned in this module:

1. Re-run all code cells with your own inputs
2. Modify one code cell to add new functionality
3. Answer the interview questions above from memory

**Mini Project**: Build a small application that combines the concepts from this module.


## 📚 Summary

In this module you learned:
- The core concepts of **Human-in-the-Loop — Pausing for Human Input**
- How to implement them with modern LangChain/LangGraph APIs
- Common mistakes and how to avoid them
- Interview-level understanding of why each component exists

**Next**: Continue to the next module to build on these foundations.

---
*Part of the LangChain & LangGraph Master Course*
